<!-- NB_DOC_INTRO_V1 -->
# MLOps — Orchestration de pipelines : Airflow, Prefect, Dagster

> 📚 **Doc thematique** : [docs/08_MLOPS.md](docs/08_MLOPS.md) (MLOps)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**Le probleme** : comment faire tourner mon pipeline ML (ingestion → preprocessing → train → eval → deploy) de maniere **reproductible, observable, retryable, schedulable** ?

Un script bash :
- ❌ Pas d'observabilite (qui a echoue, quand, pourquoi)
- ❌ Pas de retry intelligent
- ❌ Pas de parallelisme automatique
- ❌ Pas de UI pour relancer une partie du workflow

→ **Orchestrateur** = couche d'organisation des taches.

Ce notebook execute **Prefect** (recommande 2025 pour nouveau projet ML) avec un pipeline reel, et presente Airflow + Dagster en comparaison.

**Choix outils 2025** :

| Tool | Maturite | Forces | Limites |
|---|---|---|---|
| **Apache Airflow** | Tres mature (2014) | Standard industrie, enorme ecosysteme de providers | Verbose, debug local penible, DAG statique |
| **Prefect** | Bonne | Moderne, Pythonic, dynamic DAGs, debug native | Plus petit ecosysteme |
| **Dagster** | Croissante | Pensee data + asset + observabilite end-to-end | Concepts a apprendre |
| **Kedro** (QuantumBlack) | Stable | Framework projet ML (pas orchestrateur), plug a Airflow/Prefect | Pas un orchestrateur en soi |
| **Argo Workflows** | Stable | K8s native, scale | YAML lourd |
| **AWS Step Functions / GCP Workflows** | Cloud | Gere, integre | Vendor lock-in |

Versions : `prefect >= 2.14`.

## Plan

1. Installation
2. **Prefect** — concepts (flow, task, retry, cache)
3. Pipeline ML reel : ingest → preprocess → train → eval (execute en local)
4. Retries, caching, dependencies
5. Scheduling (deployment + cron)
6. **Airflow** — DAG type ML (presentation)
7. **Dagster** — data-centric assets (presentation)
8. **Kedro** — structure projet
9. Pieges et anti-patterns
10. References


## 1. Installation


In [ ]:
# pip install -q prefect scikit-learn pandas numpy
import prefect
from prefect import flow, task
from prefect.tasks import task_input_hash
from datetime import timedelta
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score

print(f"Prefect version : {prefect.__version__}")
SEED = 42

## 2. Prefect — concepts

| Concept | Sens |
|---|---|
| **`@task`** | Une unite de travail (= "noeud" du DAG). Re-tryable, cacheable, parametrable. |
| **`@flow`** | Compose des taches. Le DAG est construit a l'execution (dynamic, contrairement a Airflow). |
| **Retry** | `@task(retries=3, retry_delay_seconds=60)` |
| **Cache** | `@task(cache_key_fn=task_input_hash, cache_expiration=timedelta(hours=1))` |
| **Logger** | `from prefect import get_run_logger ; get_run_logger().info(...)` |
| **State** | Chaque task/flow a un state (Pending, Running, Completed, Failed, Cached, ...) |
| **Run** | Une execution d'un flow (a un timestamp donne) |
| **Deployment** | Flow versionne, schedulable, deployable |


## 3. Pipeline ML execute en local


In [ ]:
# === Definition des tasks ===
@task(retries=2, retry_delay_seconds=5)
def extract_data() -> pd.DataFrame:
    """Charger les donnees (ici sklearn, en realite : DB / S3 / API)."""
    data = load_breast_cancer(as_frame=True)
    df = data.frame
    print(f"  [extract] {len(df)} rows, {df.shape[1]} cols")
    return df

@task
def validate_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Validation simple (en prod : pandera, great_expectations)."""
    assert df.shape[0] > 100, "Pas assez de donnees"
    assert "target" in df.columns, "Colonne target manquante"
    print(f"  [validate] schema OK, shape={df.shape}")
    return df

@task(cache_key_fn=task_input_hash, cache_expiration=timedelta(hours=1))
def feature_engineering(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Feature eng (ici trivial, cache 1h via cache_key_fn)."""
    X = df.drop(columns="target").values
    y = df["target"].values
    print(f"  [features] X.shape={X.shape}")
    return X, y

@task
def train_model(X: np.ndarray, y: np.ndarray) -> dict:
    """Train + return metrics + model."""
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
    clf = RandomForestClassifier(n_estimators=100, random_state=SEED, n_jobs=-1).fit(X_tr, y_tr)
    proba = clf.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y_te, proba)
    f1 = f1_score(y_te, pred)
    print(f"  [train] AUC={auc:.4f}, F1={f1:.4f}")
    return {"auc": auc, "f1": f1, "model": clf, "X_test": X_te, "y_test": y_te}

@task
def evaluate(result: dict) -> dict:
    """Eval supplementaire / quality gate."""
    auc = result["auc"]
    if auc < 0.85:
        raise ValueError(f"Quality gate failed : AUC={auc:.4f} < 0.85")
    print(f"  [evaluate] Quality gate PASSED (AUC={auc:.4f})")
    return {"deploy": True, "auc": auc, "f1": result["f1"]}

@task
def deploy(decision: dict) -> str:
    """Promouvoir vers Production (placeholder)."""
    if decision["deploy"]:
        # En realite : MLflow.transition_model_version_stage, ou BentoML push, etc.
        return f"deployed (AUC={decision['auc']:.4f})"
    return "skipped"

# === Definition du flow ===
@flow(name="ml_pipeline", log_prints=True)
def ml_pipeline():
    df = extract_data()
    df_val = validate_schema(df)
    X, y = feature_engineering(df_val)
    result = train_model(X, y)
    decision = evaluate(result)
    status = deploy(decision)
    return status

# === Execute ===
status = ml_pipeline()
print(f"\nPipeline status : {status}")

**Avantage majeur du mode lazy execution Prefect** : on a vu tous les `[extract]`, `[validate]`, `[features]`, `[train]`, `[evaluate]`, `[deploy]` en sequence. Si on relance immediatement, `feature_engineering` est **cache** (gain temps + reproductibilite).


In [ ]:
# === Re-execute : la feature eng devrait etre CACHED ===
print("Second run (cache active sur feature_engineering) :")
status2 = ml_pipeline()

## 4. Retries, parallel, subflows


In [ ]:
# === Task avec retry automatique sur erreur ===
@task(retries=3, retry_delay_seconds=2)
def flaky_task(attempt_counter: list):
    attempt_counter.append(1)
    if len(attempt_counter) < 3:
        raise ConnectionError(f"Simule un fail (essai {len(attempt_counter)})")
    return f"OK apres {len(attempt_counter)} essais"

@flow(log_prints=True)
def with_retry_flow():
    attempts = []
    result = flaky_task(attempts)
    return result

print(with_retry_flow())

In [ ]:
# === Parallel : Prefect execute les taches independantes en parallele
# via .submit() (renvoie un PrefectFuture)

@task
def slow_task(name: str, duration: float = 0.5) -> str:
    import time
    time.sleep(duration)
    return f"{name} done"

@flow
def parallel_flow():
    # 3 taches lancees en parallele
    f1 = slow_task.submit("task_A")
    f2 = slow_task.submit("task_B")
    f3 = slow_task.submit("task_C")
    # Attendre les resultats
    return [f1.result(), f2.result(), f3.result()]

import time
t0 = time.perf_counter()
results = parallel_flow()
t = time.perf_counter() - t0
print(f"\nResults : {results}")
print(f"Total time : {t:.2f}s  (~ 0.5s si parallele OK, 1.5s si sequentiel)")

## 5. Scheduling — deployment + cron

Pour planifier un flow (cron), il faut le **deployer** dans Prefect server. En local :

```bash
# 1. Demarrer Prefect server (UI : http://127.0.0.1:4200)
prefect server start

# 2. Dans un autre terminal : creer un work pool
prefect work-pool create my-pool --type process

# 3. Demarrer un worker
prefect worker start --pool my-pool

# 4. Deployer le flow (dans un script Python)
```

```python
from prefect import flow

@flow
def daily_etl():
    ...

if __name__ == "__main__":
    daily_etl.serve(
        name="daily-etl-deploy",
        cron="0 2 * * *",       # tous les jours 02:00
        tags=["prod", "etl"],
        description="Daily ETL pipeline",
    )
```

**Prefect Cloud** : meme principe, deploy vers `app.prefect.cloud`. UI hostee, pas de server local a gerer.


## 6. Airflow — DAG type ML

### Concepts
- **DAG** = Directed Acyclic Graph de taches (fichier Python place dans `dags/`)
- **Operator** = type de tache (`BashOperator`, `PythonOperator`, `SqlOperator`, `S3FileTransfer`, ...)
- **Task** = instance d'operator
- **XCom** = echange de donnees entre taches (petit, < 1 MB)
- **Connection** = config externe (DB, S3, API) versionnee dans UI
- **Variable** = config simple
- **Sensor** = task qui attend une condition (fichier dans S3, partition Hive, API event)

### DAG type ML (Airflow 2 — TaskFlow API)
```python
# dags/train_model.py
from airflow.decorators import dag, task
from airflow.utils.dates import days_ago
from datetime import timedelta

@dag(
    schedule_interval="0 2 * * *",          # cron : tous les jours 02h00
    start_date=days_ago(1),
    catchup=False,
    default_args={"retries": 2, "retry_delay": timedelta(minutes=5)},
    tags=["ml", "churn"],
)
def churn_model_pipeline():

    @task
    def extract() -> str:
        return "s3://bucket/path/data.parquet"

    @task
    def validate(path: str) -> str:
        # Great Expectations / pandera
        return path

    @task
    def features(path: str) -> str:
        return "s3://bucket/features.parquet"

    @task
    def train(path: str) -> str:
        return "s3://bucket/model.pkl"

    @task
    def evaluate(model: str) -> dict:
        return {"auc": 0.92}

    @task.short_circuit                      # arrete le DAG si retour False
    def quality_gate(metrics: dict) -> bool:
        return metrics["auc"] > 0.85

    @task
    def deploy(model: str):
        ...

    data = extract()
    validated = validate(data)
    feat = features(validated)
    model = train(feat)
    metrics = evaluate(model)
    ok = quality_gate(metrics)
    deploy(model) << ok

dag = churn_model_pipeline()
```

### Lancer Airflow en local
```bash
pip install apache-airflow[postgres,celery,redis,amazon]
export AIRFLOW_HOME=$PWD/airflow_home
airflow db init
airflow users create --username admin --password admin --role Admin \
    --firstname X --lastname Y --email a@b.c
airflow webserver -p 8080 &
airflow scheduler &
# UI : http://localhost:8080
```


## 7. Dagster — data-centric (assets)

Pivot conceptuel : pas des "tasks" mais des **assets** (= tables, modeles, dashboards) avec leurs dependances. Plus aligne avec la facon dont les data scientists pensent.

```python
from dagster import asset, AssetIn
import pandas as pd

@asset
def raw_data() -> pd.DataFrame:
    return pd.read_parquet("data.parquet")

@asset(ins={"raw_data": AssetIn()})
def features(raw_data: pd.DataFrame) -> pd.DataFrame:
    return raw_data.assign(new_col=raw_data["x"] * 2)

@asset
def model(features: pd.DataFrame):
    # train
    return trained_model

# La UI Dagster montre le DAG d'ASSETS (pas de tasks abstraites)
# On peut re-materialiser UN seul asset (re-train sans re-ingest)
# Tests / docs / types attaches a chaque asset
```

Cas d'usage ou Dagster brille : **data warehouse + ML**, dbt-friendly, gestion d'assets long-lived.


## 8. Kedro — framework projet (pas orchestrateur)

Kedro n'est pas un orchestrateur, c'est un **framework de structure projet ML** :
```bash
pip install kedro
kedro new
```

Apporte :
- **Structure standard** (data/, src/, conf/, notebooks/, ...)
- **DataCatalog** : abstraction des I/O (changer S3 → local dans un YAML)
- **Pipelines reutilisables**, testables localement
- **Plug** avec Airflow / Prefect / Dagster (export du DAG vers ces outils)

Idéal : projet ML qui devient gros, plusieurs equipes, besoin de structurer code + data + config.


## 9. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| XCom avec gros datasets (1 GB+) | Ecrire sur disque/S3, passer la **reference** (path) entre taches |
| Tache qui fait trop (extract + transform + load) | Granularite fine : 1 tache = 1 responsabilite |
| Code business dans operators Airflow | Module Python externe importable, l'operator juste appel |
| Hard-coder paths / credentials | DataCatalog (Kedro) / Connections (Airflow) / Variables / Secrets |
| Pas de tests sur les pipelines | Tester chaque task isolement (mock I/O) + integration end-to-end |
| Schedule trop frequent → backfill long | Si DAG echoue plusieurs jours, `max_active_runs=1` evite cascade |
| Pas d'idempotence | Toujours pouvoir relancer SANS effet de bord (truncate/insert vs append duplique) |
| Logs caches dans stdout | Logger structure (JSON) + collecte centrale |
| Dependances implicites (l'ordre dans le code) | Toujours expliciter : `task_b(task_a())` |


## 10. References

### Documentation officielle
- **Airflow** : https://airflow.apache.org/docs/
- **Prefect** : https://docs.prefect.io/
- **Dagster** : https://docs.dagster.io/
- **Kedro** : https://docs.kedro.org/
- **Argo Workflows** : https://argoproj.github.io/argo-workflows/
- **Flyte** (alternative K8s) : https://docs.flyte.org/

### Comparatifs et tutoriels
- *Astronomer Airflow Guide* : https://www.astronomer.io/docs/
- *Prefect vs Airflow* : https://www.prefect.io/blog/airflow-vs-prefect/

### Voir aussi (collection)
- [MLOps_Tracking_DVC_Wandb.ipynb](./MLOps_Tracking_DVC_Wandb.ipynb) — tracking + versioning
- [MLOps_Model_Serving.ipynb](./MLOps_Model_Serving.ipynb) — serving prod
- [MLOps_Drift_Monitoring.ipynb](./MLOps_Drift_Monitoring.ipynb) — monitoring post-deploiement
- [SoftEng_Tests_Quality_ML.ipynb](./SoftEng_Tests_Quality_ML.ipynb) — tests, CI/CD
